In [4]:
# =================================================================
# 1. PREPARAÇÃO (Execução Silenciosa)
# =================================================================
!pip install -q "numpy<2" "scikit-fuzzy" "google-genai" matplotlib ipywidgets

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from google import genai
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configura o Matplotlib para renderização rápida
%matplotlib inline

# =================================================================
# 2. LÓGICA FUZZY (Motor de Decisão)
# =================================================================
distancia = ctrl.Antecedent(np.arange(0, 51, 1), 'distancia')
velocidade = ctrl.Antecedent(np.arange(0, 6, 1), 'velocidade')
prioridade = ctrl.Antecedent(np.arange(1, 11, 1), 'prioridade')
urgencia = ctrl.Consequent(np.arange(0, 101, 1), 'urgencia')

distancia.automf(3, names=['perto', 'media', 'longe'])
velocidade.automf(2, names=['baixa', 'alta'])
prioridade.automf(2, names=['baixa', 'alta'])
urgencia.automf(3, names=['baixa', 'media', 'alta'])

# Regras de Cobertura Universal (Evita erro de chave)
regras = [
    ctrl.Rule(distancia['perto'] | velocidade['alta'], urgencia['alta']),
    ctrl.Rule(distancia['media'] & prioridade['alta'], urgencia['alta']),
    ctrl.Rule(distancia['media'] & prioridade['baixa'], urgencia['media']),
    ctrl.Rule(distancia['longe'] & velocidade['baixa'], urgencia['baixa']),
    ctrl.Rule(prioridade['alta'] & velocidade['alta'], urgencia['alta']),
    ctrl.Rule(distancia['longe'] & prioridade['baixa'], urgencia['baixa'])
]

simulador = ctrl.ControlSystemSimulation(ctrl.ControlSystem(regras))

# Malha 3D Otimizada (15x15 para resposta instantânea)
x_range = np.linspace(0, 50, 15)
y_range = np.linspace(0, 5, 15)
x_mesh, y_mesh = np.meshgrid(x_range, y_range)
cache_z = {'last_prio': -1, 'z_mesh': None}

# =================================================================
# 3. DASHBOARD INTERATIVO DE ALTA PERFORMANCE
# =================================================================
# Sliders configurados para não gerar lag
dist_slider = widgets.IntSlider(value=15, min=0, max=50, description='Distância (m)', continuous_update=False)
vel_slider = widgets.FloatSlider(value=4.0, min=0, max=5, step=0.1, description='Velocidade (m/s)', continuous_update=False)
prio_slider = widgets.IntSlider(value=9, min=1, max=10, description='Prioridade', continuous_update=False)

btn_ia = widgets.Button(description='Gerar Relatório IA', button_style='primary', icon='rocket')
out_visual = widgets.Output()
out_txt = widgets.Output()

def renderizar_dashboard(dist, vel, prio):
    global cache_z
    with out_visual:
        clear_output(wait=True)

        # OTIMIZAÇÃO: Recalcula a superfície apenas se a prioridade mudar
        if cache_z['last_prio'] != prio:
            z_mesh = np.zeros_like(x_mesh)
            for i in range(15):
                for j in range(15):
                    simulador.input['distancia'] = x_mesh[i, j]
                    simulador.input['velocidade'] = y_mesh[i, j]
                    simulador.input['prioridade'] = prio
                    simulador.compute()
                    z_mesh[i, j] = simulador.output['urgencia']
            cache_z['z_mesh'] = z_mesh
            cache_z['last_prio'] = prio

        # Cálculo rápido do ponto atual
        simulador.input['distancia'] = dist
        simulador.input['velocidade'] = vel
        simulador.input['prioridade'] = prio
        simulador.compute()
        res = simulador.output['urgencia']

        # Visualização Profissional
        fig = plt.figure(figsize=(14, 5))
        plt.style.use('seaborn-v0_8-whitegrid')

        # 3D Plot
        ax1 = fig.add_subplot(1, 2, 1, projection='3d')
        ax1.plot_surface(x_mesh, y_mesh, cache_z['z_mesh'], cmap='plasma', alpha=0.7)
        ax1.scatter(dist, vel, res, color='black', s=150, marker='*', label='AGV')
        ax1.set_title("Estratégia de Controle 3D")
        ax1.view_init(25, 215)

        # Gauge Plot
        ax2 = fig.add_subplot(1, 2, 2)
        status_color = '#2ecc71' if res < 40 else ('#f1c40f' if res < 70 else '#e74c3c')
        ax2.barh(['URGÊNCIA'], [100], color='#ecf0f1')
        ax2.barh(['URGÊNCIA'], [res], color=status_color)
        ax2.set_xlim(0, 100)
        ax2.set_title(f"Status: {res:.1f}%")

        plt.tight_layout()
        plt.show()

def solicitar_ia(b):
    with out_txt:
        clear_output()
        print("⚡ Analisando telemetria e gerando relatório profundo...")
        CHAVE_API =  "SUA_CHAVE_AQUI"
        client = genai.Client(api_key=CHAVE_API)
        res = simulador.output['urgencia']

        prompt = f"""
        Como Engenheiro de Automação e Manutenção, analise: AGV a {dist_slider.value}m, {vel_slider.value}m/s, prioridade {prio_slider.value}.
        Urgência: {res:.2f}%.
        Gere um relatório técnico sobre:
        1. Risco de impacto cinético.
        2. Desgaste de atuadores e freios.
        3. Plano de manutenção preditiva sugerido.
        4. Uma explicação em linguagem natural sobre o porquê daquela decisão.
        5. Sugestões de ajustes ou alertas estratégicos baseados no contexto do problema.
        """
        try:
            r = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
            print("\n" + "═"*75 + "\n" + r.text + "\n" + "═"*75)
        except: print("Erro na API.")

# Conexões
btn_ia.on_click(solicitar_ia)
controles_interativos = widgets.interactive_output(renderizar_dashboard, {'dist': dist_slider, 'vel': vel_slider, 'prio': prio_slider})

# UI Layout
ui = widgets.VBox([
    widgets.HTML("<h2 style='text-align:center; color:#2c3e50;'>🛰️ Painel de Operações AGV (High Performance)</h2>"),
    widgets.HBox([dist_slider, vel_slider, prio_slider], layout=widgets.Layout(justify_content='center')),
    out_visual,
    widgets.Box([btn_ia], layout=widgets.Layout(display='flex', flex_flow='column', align_items='center', width='100%')),
    out_txt
])

display(ui)